## Parte 1 - Estrutura, Restrições e Expansão do Modelo

In [1]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_1 = _dntk.execute_sql(
  '-- Tarefa 1 - Criar a tabela \'sessoes\' com restrições de integridade\nCREATE TABLE sessoes (\n    id INTEGER PRIMARY KEY ,\n    sala TEXT NOT NULL,\n    codigo_sessao TEXT UNIQUE NOT NULL,\n    filme_id INTEGER,\n    horario TEXT NOT NULL,\n    formato TEXT DEFAULT \'Dublado\'\n);',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,Count


In [2]:
# Tarefa 2 - Analisar INSERT com NOT NULL na tabela 'atores'

import sqlite3

conn = sqlite3.connect("cinema.db")
cursor = conn.cursor()

comandos = {
    "A": "INSERT INTO atores (id, nome, pais_origem, sexo, oscar_ganhos) VALUES (51, 'Maria Fontes', 'Brasil', 'F', 0)",
    "B": "INSERT INTO atores (id, nome, pais_origem, sexo, oscar_ganhos) VALUES (52, NULL, 'EUA', 'M', 3)",
    "C": "INSERT INTO atores (id, nome, pais_origem, sexo, oscar_ganhos) VALUES (53, 'Pedro Alves', NULL, 'M', 0)"
}

for letra, sql in comandos.items():
    try:
        cursor.execute(sql)
        print(f"Comando {letra}: Sucesso")
    except Exception as e:
        print(f"Comando {letra}: Falha — {e}")

conn.rollback() # desfaz os inserts para não poluir o banco
conn.close()

Comando A: Sucesso
Comando B: Falha — NOT NULL constraint failed: atores.nome
Comando C: Sucesso


In [9]:
# Tarefa 3 - Restrição UNIQUE composta na tabela 'elenco'
import sqlite3

conn = sqlite3.connect("cinema.db")
cursor = conn.cursor()

# tenta inserir o mesmo ator no mesmo filme duas vezes
try:
    cursor.execute("INSERT INTO elenco (filme_id, ator_id, personagem) VALUES (1, 10, 'Herói')")
    cursor.execute("INSERT INTO elenco (filme_id, ator_id, personagem) VALUES (1, 10, 'Herói clone')")
    print("Ambos inseridos")
except Exception as e:
    print(f"Falha — {e}")

conn.rollback()
conn.close()

Ambos inseridos


In [11]:
# Tarefa 4 - Adicionar chaves estrangeiras à tabela 'elenco'

import sqlite3

conn = sqlite3.connect("cinema.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

cursor.executescript("""
    DROP TABLE IF EXISTS elenco;

    CREATE TABLE elenco (
        id              INTEGER PRIMARY KEY,
        filme_id        INTEGER NOT NULL,
        ator_id         INTEGER NOT NULL,
        personagem      TEXT,
        papel_principal INTEGER DEFAULT 0,
        CHECK (papel_principal IN (0, 1)),
        FOREIGN KEY (filme_id) REFERENCES filmes(id),
        FOREIGN KEY (ator_id)  REFERENCES atores(id)
    );
""")

conn.close()
print("Tabela 'elenco' recriada com FKs e CHECK!")

Tabela 'elenco' recriada com FKs e CHECK!


In [21]:
# Tarefa 5 - Modelar o sistema de ingressos 

import sqlite3
conn = sqlite3.connect("cinema.db") 
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
    CREATE TABLE IF NOT EXISTS salas (
        id         INTEGER PRIMARY KEY,
        nome       TEXT    NOT NULL,
        capacidade INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS sessoes (
        id       INTEGER PRIMARY KEY,
        sala_id  INTEGER NOT NULL,
        filme_id INTEGER NOT NULL,
        horario  TEXT    NOT NULL,
        FOREIGN KEY (sala_id)  REFERENCES salas(id),
        FOREIGN KEY (filme_id) REFERENCES filmes(id)
    );

    CREATE TABLE IF NOT EXISTS ingressos (
        id        INTEGER PRIMARY KEY,
        sessao_id INTEGER NOT NULL,
        codigo    TEXT    UNIQUE NOT NULL,
        preco     REAL,
        FOREIGN KEY (sessao_id) REFERENCES sessoes(id)
    );
""")

conn.close()
print("Tabelas 'salas', 'sessoes' e 'ingressos' criadas!")

Tabelas 'salas', 'sessoes' e 'ingressos' criadas!


In [25]:
# Tarefa 6 - Implementar novas regras de negócio via constraints
import sqlite3
conn = sqlite3.connect("cinema.db")

conn.executescript("""
    DROP TABLE IF EXISTS filmes_novo;

    CREATE TABLE filmes_novo (
        id                  INTEGER PRIMARY KEY,
        titulo              TEXT,
        ano_lancamento      INTEGER NOT NULL,
        duracao             INTEGER CHECK (duracao > 0),
        orcamento           REAL    CHECK (orcamento >= 0),
        classificacao       TEXT    CHECK (classificacao IN (
                                        'Livre', '10 anos', '12 anos',
                                        '14 anos', '16 anos', '18 anos'
                                    ))
    );
""")

conn.close()
print("Tabela 'filmes_novo' criada com todas as constraints!")

Tabela 'filmes_novo' criada com todas as constraints!


<hr>

## Parte - 2 Análise de Dados com Python

In [27]:
# Tarefa 1 - Operações entre conjuntos de diretores

diretores_critico_A = ['Greta Gerwig', 'Barry Jenkins', 'Greta Gerwig',
                        'Denis Villeneuve', 'Barry Jenkins']

diretores_critico_B = ['Christopher Nolan', 'Denis Villeneuve', 'David Fincher',
                        'Christopher Nolan', 'Denis Villeneuve']

# (a) Diretores únicos
set_A = set(diretores_critico_A)
set_B = set(diretores_critico_B)
print("Crítico A:", set_A)
print("Crítico B:", set_B)

# (b) Interseção
print("\nVistos pelos dois:", set_A & set_B)

# (c) União
print("Vistos por ao menos um:", set_A | set_B)

# (d) Diferença A - B
print("Vistos por A mas não por B:", set_A - set_B)

Crítico A: {'Denis Villeneuve', 'Barry Jenkins', 'Greta Gerwig'}
Crítico B: {'Denis Villeneuve', 'Christopher Nolan', 'David Fincher'}

Vistos pelos dois: {'Denis Villeneuve'}
Vistos por ao menos um: {'Denis Villeneuve', 'Christopher Nolan', 'Barry Jenkins', 'Greta Gerwig', 'David Fincher'}
Vistos por A mas não por B: {'Barry Jenkins', 'Greta Gerwig'}


In [29]:
# Tarefa 2 - Métodos de 'set' com categorias de premiação

categorias = ['Melhor Ator', 'Melhor Atriz', 'Melhor Fotografia', 'Melhor Ator',
              'Melhor Roteiro', 'Melhor Fotografia', 'Melhor Ator']

# (a) Set com categorias únicas
set_categorias = set(categorias)
print("Categorias únicas:", set_categorias)

# (b) Adicionar nova categoria
set_categorias.add('Melhor Direção')
print("Após add:", set_categorias)

# (c) Remover com discard
set_categorias.discard('Melhor Roteiro')
print("Após discard:", set_categorias)

# (d) Verificar se 'Melhor Atriz' está no conjunto
if 'Melhor Atriz' in set_categorias:
    print("'Melhor Atriz' está no conjunto.")
else:
    print("'Melhor Atriz' NÃO está no conjunto.")

Categorias únicas: {'Melhor Atriz', 'Melhor Fotografia', 'Melhor Ator', 'Melhor Roteiro'}
Após add: {'Melhor Ator', 'Melhor Roteiro', 'Melhor Atriz', 'Melhor Fotografia', 'Melhor Direção'}
Após discard: {'Melhor Ator', 'Melhor Atriz', 'Melhor Fotografia', 'Melhor Direção'}
'Melhor Atriz' está no conjunto.


In [31]:
# Tarefa 3 - Prêmios únicos por filme e os mais recorrentes
filmes_premios = {
    'O Último Horizonte':  ['Oscar', 'BAFTA', 'Cannes'],
    'Memórias de Outono':  ['BAFTA', 'Cannes', 'Oscar'],
    'Código Vermelho':     ['Oscar', 'BAFTA'],
    'Invasão Silenciosa':  ['Globo de Ouro', 'BAFTA'],
    'A Última Dança':      ['Globo de Ouro']
}

# (a) Todos os prêmios únicos
todos_premios = set()
for premios in filmes_premios.values():
    todos_premios.update(premios)
print("Prêmios únicos:", todos_premios)

# (b) Quantidade total
print("Total de prêmios únicos:", len(todos_premios))

# (c) Prêmios em 3 ou mais filmes
from collections import Counter

contagem = Counter()
for premios in filmes_premios.values():
    for premio in premios:
        contagem[premio] += 1

recorrentes = {p for p, n in contagem.items() if n >= 3}
print("Prêmios em 3+ filmes:", recorrentes)

Prêmios únicos: {'Globo de Ouro', 'Cannes', 'BAFTA', 'Oscar'}
Total de prêmios únicos: 4
Prêmios em 3+ filmes: {'BAFTA', 'Oscar'}


<hr>

## Parte 3 - Persistência e Intercâmbio de Dados (CSV, JSON, Pandas)

In [33]:
# Tarefa 1 - Exportar premiações para CSV e filtrar vencedores

import sqlite3
import csv

# Busca os dados
conn = sqlite3.connect("cinema.db")
cur = conn.cursor()
cur.execute("SELECT id, filme_id, premio, resultado FROM premiacoes LIMIT 15")
premiacoes = cur.fetchall()
conn.close()

# (a) Grava CSV
with open("premiacoes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "filme_id", "premio", "resultado"])
    writer.writerows(premiacoes)
print("Arquivo premiacoes.csv gerado!")

# (b) Lê e filtra vencedores
with open("premiacoes.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    vencedores = [row for row in reader if row["resultado"] == "Vencedor"]

print(f"\n{len(vencedores)} vencedor(es) encontrado(s):")
for v in vencedores:
    print(v)

Arquivo premiacoes.csv gerado!

2 vencedor(es) encontrado(s):
{'id': '5', 'filme_id': '2', 'premio': 'BAFTA', 'resultado': 'Vencedor'}
{'id': '14', 'filme_id': '3', 'premio': 'Globo de Ouro', 'resultado': 'Vencedor'}


In [37]:
# Tarefa 2 - Gravar e ler informações de filmes em JSON

import json

filmes_info = [
    {"titulo": "O Último Horizonte",  "diretor": "Christopher Nolan",  "orcamento_milhoes": 180.0},
    {"titulo": "Memórias de Outono",  "diretor": "Greta Gerwig",        "orcamento_milhoes": 45.0},
    {"titulo": "Código Vermelho",     "diretor": "David Fincher",       "orcamento_milhoes": 95.0},
    {"titulo": "Invasão Silenciosa",  "diretor": "Denis Villeneuve",    "orcamento_milhoes": 220.0},
    {"titulo": "A Última Dança",      "diretor": "Barry Jenkins",       "orcamento_milhoes": 28.0},
]

# (a) Grava JSON
with open("filmes_info.json", "w", encoding="utf-8") as f:
    json.dump(filmes_info, f, indent=2, ensure_ascii=False)
print("Arquivo filmes_info.json gerado!")

# (b) Lê e calcula média
with open("filmes_info.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

media = sum(f["orcamento_milhoes"] for f in dados) / len(dados)
print(f"Média dos orçamentos: ${media:.1f} milhões")

Arquivo filmes_info.json gerado!
Média dos orçamentos: $113.6 milhões


In [35]:
# Tarefa 3 - Leitura e escrita de premiações com Pandas

import sqlite3
import pandas as pd

conn = sqlite3.connect("cinema.db")

# (a) Lê tabela completa
df = pd.read_sql_query("SELECT * FROM premiacoes", conn)
conn.close()
print(f"Total de registros: {len(df)}")

# (b) Salva CSV sem índice
df.to_csv("premiacoes.csv", index=False, encoding="utf-8")
print("CSV salvo!")

# (c) Relê, filtra vencedores e ordena por ano decrescente
df2 = pd.read_csv("premiacoes.csv")
vencedores = df2[df2["resultado"] == "Vencedor"].sort_values("ano", ascending=False)
print(f"\n{len(vencedores)} vencedor(es):")
vencedores

Total de registros: 400
CSV salvo!

94 vencedor(es):


,id,filme_id,premio,categoria,ano,resultado
211,212,58,Cannes,Melhor Filme,2024,Vencedor
292,293,78,Venice Film Festival,Melhor Diretor,2024,Vencedor
182,183,50,Sundance,Melhor Fotografia,2024,Vencedor
104,105,26,Sundance,Melhores Efeitos Visuais,2024,Vencedor
13,14,3,Globo de Ouro,Melhor Ator,2024,Vencedor
...,...,...,...,...,...,...
30,31,6,Globo de Ouro,Melhor Roteiro Original,2020,Vencedor
32,33,6,Sundance,Melhor Trilha Sonora,2020,Vencedor
120,121,35,Oscar,Melhor Fotografia,2020,Vencedor
54,55,10,BAFTA,Melhor Fotografia,2020,Vencedor


<hr>

## Parte 4 Refatoração e Normalização do Modelo

In [39]:
# (a) Problemas identificados na tabela 'premiacoes':
problemas = """
Problema 1 — Ausência de NOT NULL nos campos essenciais:
  Campos como 'premio', 'categoria', 'ano' e 'resultado' não têm NOT NULL,
  permitindo registros incompletos sem sentido no contexto de uma premiação.

Problema 2 — Ausência de restrição no campo 'resultado':
  Sem CHECK, qualquer valor pode ser inserido em 'resultado' (ex: 'Talvez',
  'Ganhou', texto livre), comprometendo a integridade dos dados.

Problema 3 — Ausência de chave estrangeira para filme_id:
  Sem FOREIGN KEY, é possível registrar uma premiação para um filme
  inexistente, quebrando a integridade referencial.
"""
print(problemas)


Problema 1 — Ausência de NOT NULL nos campos essenciais:
  Campos como 'premio', 'categoria', 'ano' e 'resultado' não têm NOT NULL,
  permitindo registros incompletos sem sentido no contexto de uma premiação.

Problema 2 — Ausência de restrição no campo 'resultado':
  Sem CHECK, qualquer valor pode ser inserido em 'resultado' (ex: 'Talvez',
  'Ganhou', texto livre), comprometendo a integridade dos dados.

Problema 3 — Ausência de chave estrangeira para filme_id:
  Sem FOREIGN KEY, é possível registrar uma premiação para um filme
  inexistente, quebrando a integridade referencial.



In [41]:
import sqlite3

conn = sqlite3.connect("cinema.db")
conn.execute("PRAGMA foreign_keys = ON")

# (b) CREATE TABLE corrigido
conn.executescript("""
    DROP TABLE IF EXISTS premiacoes_nova;

    CREATE TABLE premiacoes_nova (
        id        INTEGER PRIMARY KEY,
        filme_id  INTEGER NOT NULL,
        premio    TEXT    NOT NULL,
        categoria TEXT    NOT NULL,
        ano       INTEGER NOT NULL,
        resultado TEXT    NOT NULL CHECK (resultado IN ('Vencedor', 'Indicado')),
        FOREIGN KEY (filme_id) REFERENCES filmes(id)
    );
""")

conn.close()
print("Tabela 'premiacoes_nova' criada com todas as restrições!")

Tabela 'premiacoes_nova' criada com todas as restrições!


In [43]:
# Tarefa 2 - Normalizar a tabela de filmes e diretores para 3FN


# Análise da desnormalização:
analise = """
Problema — Dependência transitiva (viola 3FN):
  'pais_diretor' depende de 'nome_diretor', não de 'filme_id'.
  Ou seja: filme_id → nome_diretor → pais_diretor (transitiva).

Solução — Separar em duas tabelas:
  diretores (id, nome, pais)
  filmes    (id, titulo, diretor_id FK, ano_lancamento)
"""
print(analise)


Problema — Dependência transitiva (viola 3FN):
  'pais_diretor' depende de 'nome_diretor', não de 'filme_id'.
  Ou seja: filme_id → nome_diretor → pais_diretor (transitiva).

Solução — Separar em duas tabelas:
  diretores (id, nome, pais)
  filmes    (id, titulo, diretor_id FK, ano_lancamento)



In [45]:
import sqlite3

conn = sqlite3.connect("cinema.db")
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
    DROP TABLE IF EXISTS filmes_3fn;
    DROP TABLE IF EXISTS diretores;

    CREATE TABLE diretores (
        id   INTEGER PRIMARY KEY,
        nome TEXT    NOT NULL,
        pais TEXT    NOT NULL
    );

    CREATE TABLE filmes_3fn (
        id             INTEGER PRIMARY KEY,
        titulo         TEXT    NOT NULL,
        diretor_id     INTEGER NOT NULL,
        ano_lancamento INTEGER NOT NULL,
        FOREIGN KEY (diretor_id) REFERENCES diretores(id)
    );

    -- Insere diretores únicos
    INSERT INTO diretores VALUES (1, 'Christopher Nolan', 'Reino Unido');
    INSERT INTO diretores VALUES (2, 'Greta Gerwig',      'EUA');
    INSERT INTO diretores VALUES (3, 'David Fincher',     'EUA');

    -- Insere filmes referenciando diretor_id
    INSERT INTO filmes_3fn VALUES (1, 'O Último Horizonte', 1, 2023);
    INSERT INTO filmes_3fn VALUES (2, 'Memórias de Outono', 2, 2022);
    INSERT INTO filmes_3fn VALUES (3, 'Código Vermelho',    3, 2023);
    INSERT INTO filmes_3fn VALUES (4, 'Tenet',              1, 2020);
""")

conn.commit()
conn.close()
print("Tabelas normalizadas criadas e populadas!")

Tabelas normalizadas criadas e populadas!


In [47]:
# Verifica resultado com JOIN
import sqlite3
import pandas as pd

conn = sqlite3.connect("cinema.db")

df = pd.read_sql_query("""
    SELECT f.id, f.titulo, d.nome AS diretor, d.pais, f.ano_lancamento
    FROM filmes_3fn f
    JOIN diretores d ON f.diretor_id = d.id
    ORDER BY f.id
""", conn)

conn.close()
df

,id,titulo,diretor,pais,ano_lancamento
0,1,O Último Horizonte,Christopher Nolan,Reino Unido,2023
1,2,Memórias de Outono,Greta Gerwig,EUA,2022
2,3,Código Vermelho,David Fincher,EUA,2023
3,4,Tenet,Christopher Nolan,Reino Unido,2020


In [49]:
# Tarefa 3 - Identificar e corrigir anomalias na tabela 'premiacoes'

# (a) Problemas de normalização identificados:
problemas = """
Problema 1 — Redundância de dados (viola 1FN/2FN):
  'titulo_filme', 'ano_filme' e 'pais_filme' se repetem para cada prêmio
  do mesmo filme (linhas 1, 2 e 4 repetem dados de 'O Último Horizonte').

Problema 2 — Anomalia de atualização:
  Se o país de um filme mudar, é preciso atualizar múltiplas linhas,
  arriscando inconsistência entre elas.

Problema 3 — Dependência parcial (viola 2FN):
  'titulo_filme', 'ano_filme' e 'pais_filme' dependem apenas do filme,
  não da combinação filme + prêmio que identifica cada linha.
"""
print(problemas)


Problema 1 — Redundância de dados (viola 1FN/2FN):
  'titulo_filme', 'ano_filme' e 'pais_filme' se repetem para cada prêmio
  do mesmo filme (linhas 1, 2 e 4 repetem dados de 'O Último Horizonte').

Problema 2 — Anomalia de atualização:
  Se o país de um filme mudar, é preciso atualizar múltiplas linhas,
  arriscando inconsistência entre elas.

Problema 3 — Dependência parcial (viola 2FN):
  'titulo_filme', 'ano_filme' e 'pais_filme' dependem apenas do filme,
  não da combinação filme + prêmio que identifica cada linha.



In [51]:
import sqlite3

conn = sqlite3.connect("cinema.db")
conn.execute("PRAGMA foreign_keys = ON")

# (b) Modelo normalizado
conn.executescript("""
    DROP TABLE IF EXISTS premiacoes_norm;
    DROP TABLE IF EXISTS filmes_norm;

    CREATE TABLE filmes_norm (
        id   INTEGER PRIMARY KEY,
        titulo TEXT    NOT NULL,
        ano  INTEGER NOT NULL,
        pais TEXT    NOT NULL
    );

    CREATE TABLE premiacoes_norm (
        id        INTEGER PRIMARY KEY,
        filme_id  INTEGER NOT NULL,
        premio    TEXT    NOT NULL,
        resultado TEXT    NOT NULL CHECK (resultado IN ('Vencedor', 'Indicado')),
        FOREIGN KEY (filme_id) REFERENCES filmes_norm(id)
    );

    -- Filmes únicos
    INSERT INTO filmes_norm VALUES (1, 'O Último Horizonte', 2023, 'EUA');
    INSERT INTO filmes_norm VALUES (2, 'Memórias de Outono', 2022, 'EUA');

    -- Premiações sem repetir dados do filme
    INSERT INTO premiacoes_norm VALUES (1, 1, 'Oscar',  'Indicado');
    INSERT INTO premiacoes_norm VALUES (2, 1, 'BAFTA',  'Indicado');
    INSERT INTO premiacoes_norm VALUES (3, 2, 'BAFTA',  'Vencedor');
    INSERT INTO premiacoes_norm VALUES (4, 1, 'Cannes', 'Indicado');
""")

conn.commit()
conn.close()
print("Modelo normalizado criado!")

Modelo normalizado criado!


In [53]:
# Verifica resultado com JOIN
import sqlite3
import pandas as pd

conn = sqlite3.connect("cinema.db")

df = pd.read_sql_query("""
    SELECT f.titulo, f.ano, f.pais, p.premio, p.resultado
    FROM premiacoes_norm p
    JOIN filmes_norm f ON p.filme_id = f.id
    ORDER BY f.titulo, p.premio
""", conn)

conn.close()
df

,titulo,ano,pais,premio,resultado
0,Memórias de Outono,2022,EUA,BAFTA,Vencedor
1,O Último Horizonte,2023,EUA,BAFTA,Indicado
2,O Último Horizonte,2023,EUA,Cannes,Indicado
3,O Último Horizonte,2023,EUA,Oscar,Indicado


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b0488197-ad08-41ff-a0a7-ce71b5c7daaa' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>